
# Text-to-SQL：让大模型写 SQL，再交给数据库执行

流程：自然语言问题 → 把表结构塞进 Prompt → 大模型生成 SQL → 清洗 → 交给 SQLite 真执行 → 拿回结果。

**关键点**：大模型只负责"写"，不负责"对"。SQL 到底对不对，要用真实数据库跑一遍才知道——
这正是这个项目比"让 ChatGPT 写段 SQL"多出来的部分。

> 依赖上一个 notebook 生成的 `company.db`。如果文件不存在，先运行 `01_sql_basics.ipynb`。



## 1. 配置客户端

API Key 从环境变量读，不要写死在代码里——写死之后一旦把 notebook 传到 GitHub，Key 就泄露了。

```bash
# Windows PowerShell
$env:DEEPSEEK_API_KEY = "sk-..."
# macOS / Linux
export DEEPSEEK_API_KEY="sk-..."
```


In [ ]:

import os

from openai import OpenAI

api_key = os.environ.get("DEEPSEEK_API_KEY")
if not api_key:
    raise SystemExit("请先设置环境变量 DEEPSEEK_API_KEY")

# DeepSeek 的接口兼容 OpenAI SDK，换个 base_url 即可
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")



## 2. 把表结构告诉模型

大模型不知道你的数据库长什么样，所以要在 Prompt 里给出表名和字段说明。


In [ ]:

table_schema = '''
表名: employees
字段:
- id (员工编号)
- name (姓名)
- department (部门)
- salary (工资)
'''

user_question = "帮我查一下研发部工资大于 15000 的人有哪些？"

prompt = (
    "你是一个专业的 SQL 数据分析师。
"
    f"这是我们的数据库表结构：
{table_schema}

"
    "请根据我的问题，写出一条 SQL 查询语句。
"
    "不要输出任何多余的解释，只输出 SQL 代码。

"
    f"我的问题是：{user_question}"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": prompt}],
)
sql_query = response.choices[0].message.content
print(sql_query)



模型经常会把 SQL 包在 Markdown 代码块里（```` ```sql ... ``` ````），执行前必须清理掉。

预期输出大致是：

```sql
SELECT id, name, department, salary
FROM employees
WHERE department = '研发部' AND salary > 15000;
```


In [ ]:

import sqlite3

import pandas as pd

clean_sql = sql_query.replace("```sql", "").replace("```", "").strip()
print("清理后的 SQL：")
print(clean_sql)



## 3. 执行并验证

模型写的 SQL 不一定对：字段名可能编错、方言可能不兼容。只有真跑一遍才知道。


In [ ]:

conn = sqlite3.connect("company.db")
try:
    result = pd.read_sql_query(clean_sql, conn)
    print("=== 数据库真实返回 ===")
    print(result)
except Exception as exc:
    print(f"SQL 执行失败：{exc}")
finally:
    conn.close()



## 小结

- 大模型生成的是**文本**，不是结果，必须落到真实数据库上验证；
- 表结构写得越清楚（字段名 + 含义），生成的 SQL 越靠谱；
- 生产环境还要加一道闸：只允许 `SELECT`、限制 `LIMIT`、用只读账号连接——
  否则模型完全可能生成 `DROP TABLE`。

**可以继续做**

1. 让模型在 SQL 报错时自动重试（把报错信息回灌给模型）；
2. 支持多轮：用户追问"只看工资最高的前三个"时，模型基于上一轮的 SQL 修改；
3. 用 `sqlglot` 做 SQL 语法校验，挡掉明显越权的语句。
